# Parte B – Técnicas Aproximadas de Control por Refuerzo

**Grupo 2 · MIA UMU 2025/2026**

---

## Objetivo

Implementar, entrenar y comparar dos algoritmos de control aproximado:

| Algoritmo | Tipo | Aproximador |
|-----------|------|-------------|
| **SARSA semi-gradiente** | On-policy | Lineal (base de Fourier / features artesanales) |
| **Deep Q-Network (DQN)** | Off-policy | Red neuronal (MLP) |

### Entornos

* **CartPole-v1** (Gymnasium) — espacio continuo 4D, referencia de calibración.
* **Tetris** (`tetris_gymnasium/Tetris`) — tablero 24×18, espacio colosal;
  evidencia la necesidad de aproximación funcional.

### ¿Por qué aproximación funcional?

En CartPole el estado es ${R}^4$, continuo e infinito: imposible tabular.  
En Tetris la representación naïve del tablero tiene >$2^{200}$ configuraciones posibles,
haciendo inviable cualquier tabla Q discreta. La aproximación es la única alternativa viable.

---

> **Reproducibilidad**: semillas fijadas globalmente; corre completo con *Ejecutar todas* en Colab.

In [1]:
# Instalación de dependencias (ejecutar en Colab o entorno limpio)
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('gymnasium[classic-control]>=0.29')
pip('tetris-gymnasium')
pip('torch>=2.0')
pip('matplotlib>=3.7')
pip('numpy>=1.24')
pip('pandas>=2.0')

print('Dependencias instaladas.')

Dependencias instaladas.


In [ ]:
import os, sys, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import gymnasium as gym
import torch, torch.nn as nn, torch.optim as optim
from collections import deque
from itertools import product
from IPython.display import HTML, display
from matplotlib import animation

# Se define la configuración experimental común para entrenar y comparar ambos algoritmos.
EXP_MODE = os.getenv('EXP_MODE', 'FULL').upper()
if EXP_MODE not in {'FAST', 'FULL'}:
    raise ValueError(f"EXP_MODE invalido: {EXP_MODE}. Usa FAST o FULL.")

if EXP_MODE == 'FULL':
    SEEDS            = [0, 1, 2, 3, 4]
    N_EP_CARTPOLE    = 500
    # Se incrementa Tetris para una muestra temporal mas representativa.
    N_EP_TETRIS      = 250
    MAX_STEPS_TETRIS = 800
    SMOOTH_W         = 30
    W_FINAL_CARTPOLE = 50
    W_FINAL_TETRIS   = 40
    EVAL_SEEDS_CARTPOLE = [10_000 + i for i in range(25)]
    EVAL_SEEDS_TETRIS   = [20_000 + i for i in range(20)]
else:
    # FAST: pensado para validar rapido en CPU/Colab sin cortar ejecucion.
    SEEDS            = [0, 1, 2]
    N_EP_CARTPOLE    = 250
    N_EP_TETRIS      = 80
    MAX_STEPS_TETRIS = 500
    SMOOTH_W         = 20
    W_FINAL_CARTPOLE = 30
    W_FINAL_TETRIS   = 20
    EVAL_SEEDS_CARTPOLE = [10_000 + i for i in range(20)]
    EVAL_SEEDS_TETRIS   = [20_000 + i for i in range(10)]

# Programa de epsilon para SARSA: decay calculado para llegar exactamente a eps_end
# en el último episodio del presupuesto. Formula: eps_decay = eps_end ^ (1 / T)
# Esto evita que SARSA desperdicie exploracion (decay muy lento) o la consuma
# demasiado pronto (decay muy agresivo) independientemente del modo FAST/FULL.
SARSA_EPS_DECAY_CP  = float(0.01 ** (1.0 / N_EP_CARTPOLE))   # eps_end=0.01 para CartPole
SARSA_EPS_DECAY_TET = float(0.05 ** (1.0 / N_EP_TETRIS))     # eps_end=0.05 para Tetris

MAX_STEPS_CARTPOLE = 500

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch  : {torch.__version__}')
print(f'Gymnasium: {gym.__version__}')
print(f'Device   : {DEVICE}')
print(f'Modo     : {EXP_MODE}')
print(f'Train seeds: {SEEDS}')
print(f'CartPole: {N_EP_CARTPOLE} ep | Tetris: {N_EP_TETRIS} ep')
print(f'Eval seeds CartPole/Tetris: {len(EVAL_SEEDS_CARTPOLE)}/{len(EVAL_SEEDS_TETRIS)}')
print(f'SARSA eps_decay CartPole: {SARSA_EPS_DECAY_CP:.5f}  ({N_EP_CARTPOLE} ep -> eps_end=0.01)')
print(f'SARSA eps_decay Tetris  : {SARSA_EPS_DECAY_TET:.5f}  ({N_EP_TETRIS} ep -> eps_end=0.05)')


PyTorch  : 2.10.0+cu128
Gymnasium: 1.2.3
Device   : cuda
Modo     : FULL
Train seeds: [0, 1, 2, 3, 4]
CartPole: 500 ep | Tetris: 250 ep
Eval seeds CartPole/Tetris: 25/20
SARSA eps_decay CartPole: 0.99083  (500 ep -> eps_end=0.01)
SARSA eps_decay Tetris  : 0.98809  (250 ep -> eps_end=0.05)


In [ ]:
# Se fijan semillas y opciones deterministas para garantizar reproducibilidad experimental.
def set_global_seed(seed: int):
    """Semillas en Python/NumPy/Torch + ajustes deterministas."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


def episode_seed(base_seed: int, episode_idx: int) -> int:
    """Se usa la misma semilla por episodio para comparar SARSA y DQN bajo condiciones equivalentes."""
    return int(base_seed * 100_000 + episode_idx)


set_global_seed(SEEDS[0])
print(f'Semilla global fijada en {SEEDS[0]}.')


In [ ]:
# Se verifica CartPole para confirmar dimensiones de entrada y acciones disponibles.
_cp = gym.make('CartPole-v1')
_obs, _ = _cp.reset(seed=0)
print('CartPole-v1')
print('  obs shape   :', _obs.shape)
print('  action space:', _cp.action_space)
_cp.close()

# Se verifica Tetris para confirmar forma de tablero y número de acciones.
TETRIS_AVAILABLE = False
TETRIS_ENV_ID    = 'tetris_gymnasium/Tetris'
TETRIS_IN_DIM    = None
TETRIS_N_ACTIONS = None

try:
    from tetris_gymnasium.envs import Tetris as _TetrisEnvClass  # registra el env
    _tet = gym.make(TETRIS_ENV_ID)
    _obs_tet, _ = _tet.reset(seed=0)
    _board = _obs_tet['board'] if isinstance(_obs_tet, dict) else np.asarray(_obs_tet)
    _board = np.asarray(_board, dtype=np.float32)
    if _board.ndim > 2:
        _board = _board.squeeze()
    TETRIS_IN_DIM    = int(_board.flatten().shape[0])
    TETRIS_N_ACTIONS = int(_tet.action_space.n)
    TETRIS_AVAILABLE = True
    print()
    print('Tetris (' + TETRIS_ENV_ID + ')')
    print('  board shape :', _board.shape)
    print('  action space:', _tet.action_space)
    print('  flat in_dim :', TETRIS_IN_DIM)
    _tet.close()
except Exception as _e:
    print()
    print('Tetris no disponible:', _e)
    print('Los experimentos con Tetris seran omitidos.')


### Cierre de Infraestructura

Decisiones tomadas y razón:

- Se valida disponibilidad de ambos entornos antes de entrenar para evitar ejecuciones parciales ambiguas.
- Se centraliza configuración (`EXP_MODE`, semillas, episodios, ventanas) para reproducibilidad.
- Se deja `FAST` para validar pipeline y `FULL` para resultados finales reportables.


---
## 1. SARSA Semi-Gradiente

### Fundamento teórico

SARSA semi-gradiente (Sutton & Barto, §10.1) parametriza la función de valor acción:

$$Q(s, a; \mathbf{w}) = \mathbf{w}_a^\top \boldsymbol{\phi}(s)$$

y aplica la actualización:

$$\mathbf{w}_a \leftarrow \mathbf{w}_a + \alpha \,\underbrace{\bigl[r + \gamma\, Q(s', a'; \mathbf{w}) - Q(s, a; \mathbf{w})\bigr]}_{\delta_t} \boldsymbol{\phi}(s)$$

### Características

**CartPole**: Base de Fourier coseno de orden 2 ($3^4 = 81$ features)  
con escalas $\alpha_i = \alpha / \|\mathbf{c}_i\|$ (Konidaris et al., 2011).

**Tetris**: 7 features artesanales del tablero  
(altura agregada, altura máxima, huecos, irregularidad, desv. alturas, líneas completas, pozos).

### Control de comparabilidad (mismas condiciones)

Durante entrenamiento, cada episodio usa una semilla determinista `episode_seed(seed, ep)`
compartida por SARSA y DQN. La comparación final además se realiza con una batería fija
de evaluación (`epsilon=0`) usando exactamente las mismas semillas de evaluación.


In [5]:
# Se construye una base de Fourier para aproximar valores en el estado continuo de CartPole.
CARTPOLE_LOW  = np.array([-4.8, -3.5, -0.42, -3.5], dtype=np.float64)
CARTPOLE_HIGH = np.array([ 4.8,  3.5,  0.42,  3.5], dtype=np.float64)

class FourierFeatures:
    """phi_i(s) = cos(pi * c_i . s_norm), s_norm in [0,1]^n."""
    def __init__(self, obs_low, obs_high, order=2):
        self.low = np.asarray(obs_low,  dtype=np.float64)
        self.rng = np.asarray(obs_high, dtype=np.float64) - self.low + 1e-8
        indices  = list(product(range(order + 1), repeat=len(self.low)))
        self.C   = np.array(indices, dtype=np.float64)       # (N, n_dims)
        self.n_features = len(self.C)

    def __call__(self, obs):
        s      = np.clip(np.asarray(obs, dtype=np.float64), self.low, self.low + self.rng)
        s_norm = (s - self.low) / self.rng
        return np.cos(np.pi * (self.C @ s_norm))

    def alpha_scaling(self):
        norms = np.linalg.norm(self.C, axis=1)
        norms[norms == 0] = 1.0
        return 1.0 / norms

# Se define un extractor de variables resumen para reducir la dimensionalidad del tablero en Tetris.
N_TETRIS_FEAT = 7

def _get_board(obs):
    if isinstance(obs, dict):
        board = obs.get('board', list(obs.values())[0])
    else:
        board = obs
    board = np.asarray(board, dtype=np.float32)
    if board.ndim > 2:
        board = board.squeeze()
    return board

def tetris_featurize(obs):
    """7 features normalizadas del tablero."""
    board = _get_board(obs)
    H, W  = board.shape
    heights = np.zeros(W, dtype=np.float32)
    for c in range(W):
        filled = np.where(board[:, c] > 0)[0]
        if len(filled) > 0:
            heights[c] = H - filled[0]
    agg_h    = float(heights.sum())
    max_h    = float(heights.max())
    bumpy    = float(np.abs(np.diff(heights)).sum())
    std_h    = float(heights.std())
    complete = float(np.all(board > 0, axis=1).sum())
    holes = 0.0
    for c in range(W):
        found_top = False
        for cell in board[:, c]:
            if cell > 0:
                found_top = True
            elif found_top:
                holes += 1
    wells = 0.0
    for c in range(W):
        left  = heights[c - 1] if c > 0     else float(H)
        right = heights[c + 1] if c < W - 1 else float(H)
        depth = min(left, right) - heights[c]
        if depth > 0:
            wells += depth
    divs = np.array([H*W, H, H*W/4.0, H*W, H, 4.0, H*W], dtype=np.float32) + 1e-8
    raw  = np.array([agg_h, max_h, holes, bumpy, std_h, complete, wells], dtype=np.float32)
    return np.clip(raw / divs, 0.0, 1.0)

_ff = FourierFeatures(CARTPOLE_LOW, CARTPOLE_HIGH, order=2)
print(f'FourierFeatures order=2 -> {_ff.n_features} features')
print(f'Tetris features -> {N_TETRIS_FEAT} values')

FourierFeatures order=2 -> 81 features
Tetris features -> 7 values


In [ ]:
# Se implementa el agente SARSA lineal con política epsilon-greedy y update semi-gradiente.
class SarsaLinearAgent:
    """
    SARSA semi-gradiente lineal: Q(s,a;w) = w_a . phi(s)
    Política epsilon-greedy con decaimiento multiplicativo por episodio.
    """
    def __init__(self, n_features, n_actions, alpha=5e-3, gamma=0.99,
                 eps_start=1.0, eps_end=0.01, eps_decay=0.995, alpha_scales=None):
        self.n_actions = n_actions
        self.gamma     = gamma
        self.epsilon   = eps_start
        self.eps_end   = eps_end
        self.eps_decay = eps_decay
        # Se inicializa una fila de pesos por acción, cada una sobre el vector de features.
        self.weights   = np.zeros((n_actions, n_features), dtype=np.float64)
        # Se permite escalar alpha por feature cuando la base de Fourier lo requiere.
        self.alpha = (alpha * alpha_scales.astype(np.float64)
                      if alpha_scales is not None else float(alpha))

    def q_values(self, phi):
        return self.weights @ phi

    def select_action(self, phi):
        if np.random.random() < self.epsilon:
            return int(np.random.randint(self.n_actions))
        return int(np.argmax(self.q_values(phi)))

    def update(self, phi, action, reward, phi_next, next_action, done):
        q_cur  = float(self.weights[action] @ phi)
        q_next = 0.0 if done else float(self.weights[next_action] @ phi_next)
        delta  = reward + self.gamma * q_next - q_cur
        self.weights[action] += self.alpha * delta * phi
        return delta

    def decay_epsilon(self):
        self.epsilon = max(self.eps_end, self.epsilon * self.eps_decay)

print('SarsaLinearAgent definido.')


In [7]:
def train_sarsa_cartpole(seed=0, n_episodes=N_EP_CARTPOLE,
                         alpha=3e-3, gamma=0.99,
                         eps_start=1.0, eps_end=0.01, eps_decay=SARSA_EPS_DECAY_CP,
                         fourier_order=2, max_steps=MAX_STEPS_CARTPOLE,
                         return_agent=False):
    set_global_seed(seed)
    env  = gym.make('CartPole-v1')
    env.action_space.seed(seed)
    if hasattr(env.observation_space, 'seed'):
        env.observation_space.seed(seed)

    feat = FourierFeatures(CARTPOLE_LOW, CARTPOLE_HIGH, order=fourier_order)
    # alpha + escalado Fourier + gamma/eps elegidos para estabilidad on-policy en estado continuo.
    agent = SarsaLinearAgent(
        feat.n_features, env.action_space.n,
        alpha=alpha, gamma=gamma,
        eps_start=eps_start, eps_end=eps_end, eps_decay=eps_decay,
        alpha_scales=feat.alpha_scaling()
    )

    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=episode_seed(seed, ep))
        phi    = feat(obs)
        action = agent.select_action(phi)
        ep_ret, done, steps = 0.0, False, 0
        while not done and steps < max_steps:
            next_obs, r, term, trunc, _ = env.step(action)
            done       = term or trunc
            phi_next   = feat(next_obs)
            next_action = 0 if done else agent.select_action(phi_next)
            agent.update(phi, action, r, phi_next, next_action, done)
            ep_ret += r
            phi, action = phi_next, next_action
            steps += 1
        agent.decay_epsilon()
        returns.append(ep_ret)

    env.close()
    if return_agent:
        return returns, agent, feat
    return returns


def train_sarsa_tetris(seed=0, n_episodes=N_EP_TETRIS,
                       alpha=1e-2, gamma=0.99,
                       eps_start=1.0, eps_end=0.05, eps_decay=SARSA_EPS_DECAY_TET,
                       max_steps=MAX_STEPS_TETRIS, return_agent=False):
    set_global_seed(seed)
    from tetris_gymnasium.envs import Tetris as _TetrisEnvClass  # registra env
    env   = gym.make(TETRIS_ENV_ID)
    env.action_space.seed(seed)
    if hasattr(env.observation_space, 'seed'):
        env.observation_space.seed(seed)

    # En Tetris se sube alpha y eps_end para compensar reward escasa y mayor dificultad.
    agent = SarsaLinearAgent(
        N_TETRIS_FEAT, env.action_space.n,
        alpha=alpha, gamma=gamma,
        eps_start=eps_start, eps_end=eps_end, eps_decay=eps_decay
    )

    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=episode_seed(seed, ep))
        phi    = tetris_featurize(obs)
        action = agent.select_action(phi)
        ep_ret, done, steps = 0.0, False, 0
        while not done and steps < max_steps:
            next_obs, r, term, trunc, _ = env.step(action)
            done        = term or trunc
            phi_next    = tetris_featurize(next_obs)
            next_action = 0 if done else agent.select_action(phi_next)
            agent.update(phi, action, r, phi_next, next_action, done)
            ep_ret += r
            phi, action = phi_next, next_action
            steps += 1
        agent.decay_epsilon()
        returns.append(ep_ret)

    env.close()
    if return_agent:
        return returns, agent
    return returns


def evaluate_sarsa_cartpole(agent, feat, eval_seeds=EVAL_SEEDS_CARTPOLE,
                            max_steps=MAX_STEPS_CARTPOLE):
    env = gym.make('CartPole-v1')
    rets = []
    for s in eval_seeds:
        obs, _ = env.reset(seed=s)
        phi = feat(obs)
        done, ep_ret, steps = False, 0.0, 0
        while not done and steps < max_steps:
            action = int(np.argmax(agent.q_values(phi)))
            next_obs, r, term, trunc, _ = env.step(action)
            done = term or trunc
            ep_ret += r
            phi = feat(next_obs)
            steps += 1
        rets.append(ep_ret)
    env.close()
    return np.array(rets, dtype=float)


def evaluate_sarsa_tetris(agent, eval_seeds=EVAL_SEEDS_TETRIS,
                          max_steps=MAX_STEPS_TETRIS):
    from tetris_gymnasium.envs import Tetris as _TetrisEnvClass  # registra env
    env = gym.make(TETRIS_ENV_ID)
    rets = []
    for s in eval_seeds:
        obs, _ = env.reset(seed=s)
        phi = tetris_featurize(obs)
        done, ep_ret, steps = False, 0.0, 0
        while not done and steps < max_steps:
            action = int(np.argmax(agent.q_values(phi)))
            next_obs, r, term, trunc, _ = env.step(action)
            done = term or trunc
            ep_ret += r
            phi = tetris_featurize(next_obs)
            steps += 1
        rets.append(ep_ret)
    env.close()
    return np.array(rets, dtype=float)


print('Funciones SARSA definidas (train + eval fija).')


Funciones SARSA definidas (train + eval fija).


### Implementación SARSA

Diseño:

- Aproximador lineal para mantener trazabilidad del update semi-gradiente.
- `episode_seed(seed, ep)` para exponer ambos algoritmos a episodios equivalentes.
- Evaluación separada (`epsilon=0`) para medir política aprendida sin ruido de exploración.


### 1.1 SARSA en CartPole-v1

Se ejecuta con múltiples semillas (`SEEDS`) para estimar variabilidad.
La curva muestra media móvil (ventana `SMOOTH_W`) con banda ±1σ entre semillas.
La comparación final no depende solo de train: se usa también evaluación fija `epsilon=0`.


In [ ]:
# Se entrena SARSA en CartPole para todas las semillas de entrenamiento.
print(f'Entrenando SARSA CartPole ({N_EP_CARTPOLE} ep x {len(SEEDS)} semillas)...')
sarsa_cp_all, sarsa_cp_eval = [], []
sarsa_cp_demo_agent, sarsa_cp_demo_feat = None, None

for s in SEEDS:
    print(f'  seed={s}...', end=' ', flush=True)
    rets, agent, feat = train_sarsa_cartpole(seed=s, eps_decay=SARSA_EPS_DECAY_CP, return_agent=True)
    # La evaluación fija sin exploración permite comparar políticas finales en condiciones equivalentes.
    eval_rets = evaluate_sarsa_cartpole(agent, feat)
    print(f'train-ult{W_FINAL_CARTPOLE}={np.mean(rets[-W_FINAL_CARTPOLE:]):.1f} | eval={np.mean(eval_rets):.1f}')
    sarsa_cp_all.append(rets)
    sarsa_cp_eval.append(eval_rets)
    if s == SEEDS[0]:
        sarsa_cp_demo_agent, sarsa_cp_demo_feat = agent, feat

sarsa_cp_all  = np.array(sarsa_cp_all)   # (n_seeds, n_episodes)
sarsa_cp_eval = np.array(sarsa_cp_eval)  # (n_seeds, n_eval_episodes)
print('Listo.')


In [ ]:
# Se define un suavizado por media móvil para interpretar mejor tendencia de aprendizaje.
def smooth(arr, w=SMOOTH_W):
    if len(arr) < w:
        return np.array(arr, dtype=float)
    return np.convolve(arr, np.ones(w) / w, mode='valid')


def plot_band(ax, returns_all, color, label, w=SMOOTH_W):
    sm   = np.array([smooth(r, w) for r in returns_all])
    n    = min(len(s) for s in sm)
    sm   = np.vstack([s[:n] for s in sm])
    mean, std = sm.mean(0), sm.std(0)
    x = np.arange(n)
    ax.plot(x, mean, color=color, label=label, linewidth=1.8)
    ax.fill_between(x, mean - std, mean + std, alpha=0.25, color=color)


# Se visualiza entrenamiento de SARSA en CartPole y se reporta resumen train/eval.
fig, ax = plt.subplots(figsize=(9, 4))
plot_band(ax, sarsa_cp_all, '#1f77b4', 'SARSA semi-grad.')
ax.axhline(195, color='gray', linestyle='--', linewidth=0.9, label='Umbral (195)')
ax.axhline(500, color='green', linestyle=':', linewidth=0.9, label='Max (500)')
ax.set_xlabel('Episodio')
ax.set_ylabel(f'Retorno (MA-{SMOOTH_W})')
ax.set_title('SARSA Semi-Gradiente - CartPole-v1 (media +/- sigma, train)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Train media (ult.{W_FINAL_CARTPOLE} ep): {sarsa_cp_all[:, -W_FINAL_CARTPOLE:].mean():.1f} +/- {sarsa_cp_all[:, -W_FINAL_CARTPOLE:].std():.1f}')
print(f'Eval fija eps=0 (mismas seeds): {sarsa_cp_eval.mean():.1f} +/- {sarsa_cp_eval.std():.1f}')


### Resultados SARSA CartPole (FULL)

- **Train final (ult. 50 episodios):** `500.0 ± 0.0`.
- **Eval fija eps=0:** `500.0 ± 0.0`.
- **Semillas (5/5):** todas convergen al máximo del entorno.
- SARSA con Fourier resolvió CartPole de forma completamente estable en este setup.
- No hay gap train/eval, lo que indica que la política aprendida generaliza bien en la batería fija.


### 1.2 SARSA en Tetris

En Tetris, la representación lineal con 7 features artesanales comprime enormemente el estado.
El agente debe aprender a valorar posiciones de tablero con capacidad representacional limitada.
Se espera mejora modesta sobre política aleatoria, con aprendizaje lento.

In [ ]:
# Se entrena SARSA en Tetris para todas las semillas y luego se evalúa sin exploración.
sarsa_tet_all, sarsa_tet_eval = None, None
sarsa_tet_demo_agent = None

if TETRIS_AVAILABLE:
    print(f'Entrenando SARSA Tetris ({N_EP_TETRIS} ep x {len(SEEDS)} semillas)...')
    _tmp_train, _tmp_eval = [], []
    for s in SEEDS:
        print(f'  seed={s}...', end=' ', flush=True)
        rets, agent = train_sarsa_tetris(seed=s, eps_decay=SARSA_EPS_DECAY_TET, return_agent=True)
        eval_rets = evaluate_sarsa_tetris(agent)
        print(f'train-ult{W_FINAL_TETRIS}={np.mean(rets[-W_FINAL_TETRIS:]):.3f} | eval={np.mean(eval_rets):.3f}')
        _tmp_train.append(rets)
        _tmp_eval.append(eval_rets)
        if s == SEEDS[0]:
            sarsa_tet_demo_agent = agent

    sarsa_tet_all  = np.array(_tmp_train)
    sarsa_tet_eval = np.array(_tmp_eval)
    print('ok.')

In [ ]:
# Se visualiza entrenamiento de SARSA en Tetris y se reporta resumen train/eval.
if sarsa_tet_all is not None:
    fig, ax = plt.subplots(figsize=(9, 4))
    plot_band(ax, sarsa_tet_all, '#1f77b4', 'SARSA semi-grad.')
    ax.set_xlabel('Episodio')
    ax.set_ylabel(f'Retorno (MA-{SMOOTH_W})')
    ax.set_title('SARSA Semi-Gradiente - Tetris (media +/- sigma, train)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f'Train media (ult.{W_FINAL_TETRIS} ep): {sarsa_tet_all[:, -W_FINAL_TETRIS:].mean():.3f} +/- {sarsa_tet_all[:, -W_FINAL_TETRIS:].std():.3f}')
    print(f'Eval fija eps=0 (mismas seeds): {sarsa_tet_eval.mean():.3f} +/- {sarsa_tet_eval.std():.3f}')
else:
    print('Sin datos de Tetris.')


### Resultados SARSA Tetris (FULL)

- **Train final (ult. 40 episodios):** `9.785 ± 0.537`.
- **Eval fija eps=0 (agregada):** `9.950 ± 0.589`.
- En tabla (media por semilla), SARSA queda en **9.95** en las 5 semillas.
- SARSA lineal mantiene comportamiento muy estable pero con techo bajo.
- El rendimiento es consistente entre semillas, coherente con un aproximador de baja capacidad.


## 2. Deep Q-Network (DQN)

### Fundamento teórico

DQN (Mnih et al., 2015) parametriza $Q(s,\cdot;\theta)$ con una red neuronal y añade:

1. **Replay buffer**: muestreo aleatorio de transiciones para romper correlación temporal.
2. **Target network** $Q_{\text{target}}$ con parámetros congelados para estabilizar el objetivo:

$$y_i = r_i + \gamma\,(1-d_i)\,\max_{a'}Q_{\text{target}}(s'_i,a';\theta^-)$$

$$\mathcal{L}(\theta) = \mathbb{E}[\text{Huber}(Q(s_i,a_i;\theta) - y_i)]$$

Además: gradient clipping ($\|\nabla\|_2 \leq 10$).

| Entorno | Input | Capas | Output |
|---------|-------|-------|--------|
| CartPole | 4 | 128x128 | 2 |
| Tetris | $H \times W$ (plano) | 256x256 | n_acciones |

**Comparabilidad**: DQN usa exactamente las mismas semillas de episodio y la misma
batería de evaluación fija (`epsilon=0`) que SARSA.


In [ ]:
# Se define un buffer de experiencia para DQN con muestreo uniforme sin reemplazo.
class ReplayBuffer:
    """Buffer de experiencias con muestreo uniforme sin reemplazo."""
    def __init__(self, capacity=20_000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((
            np.asarray(state,      dtype=np.float32),
            int(action), float(reward),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        ))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.tensor(np.array(s),  dtype=torch.float32),
            torch.tensor(np.array(a),  dtype=torch.int64),
            torch.tensor(np.array(r),  dtype=torch.float32),
            torch.tensor(np.array(ns), dtype=torch.float32),
            torch.tensor(np.array(d),  dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buf)

print('ReplayBuffer definido.')


In [ ]:
# Se usa una MLP configurable para aproximar Q(s,·) en DQN.
class QNetwork(nn.Module):
    """MLP: input -> [hidden x n_layers] -> output."""
    def __init__(self, in_dim, out_dim, hidden=128, n_layers=2):
        super().__init__()
        layers, d = [], in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.ReLU()]
            d = hidden
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print('QNetwork definido.')


In [ ]:
# Se implementa el agente DQN con red online, red objetivo y replay buffer.
class DQNAgent:
    """
    Deep Q-Network con:
      - Q-network online + target network (actualización periódica)
      - Replay buffer con muestreo uniforme
      - Pérdida Huber (SmoothL1) + gradient clipping
      - Política epsilon-greedy con decaimiento multiplicativo
    """
    def __init__(self, input_dim, n_actions, hidden=128, n_layers=2,
                 lr=5e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.01, eps_decay=0.9995,
                 buf_size=20_000, batch=64, target_freq=200, grad_clip=10.0):
        self.n_actions   = n_actions
        self.gamma       = gamma
        self.epsilon     = eps_start
        self.eps_end     = eps_end
        self.eps_decay   = eps_decay
        self.batch       = batch
        self.target_freq = target_freq
        self.grad_clip   = grad_clip
        self._steps      = 0

        # La red online se actualiza en cada paso de aprendizaje.
        self.q_net       = QNetwork(input_dim, n_actions, hidden, n_layers).to(DEVICE)
        # La red objetivo proporciona targets más estables para evitar oscilaciones.
        self.target_net  = QNetwork(input_dim, n_actions, hidden, n_layers).to(DEVICE)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.loss_fn   = nn.SmoothL1Loss()
        self.buffer    = ReplayBuffer(buf_size)

    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return int(np.random.randint(self.n_actions))
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return int(self.q_net(s).argmax(1).item())

    def store(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)

    def learn(self):
        if len(self.buffer) < self.batch:
            return None
        s, a, r, ns, d = [t.to(DEVICE) for t in self.buffer.sample(self.batch)]
        q_pred = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next   = self.target_net(ns).max(dim=1).values
            q_target = r + self.gamma * q_next * (1.0 - d)

        loss = self.loss_fn(q_pred, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), self.grad_clip)
        self.optimizer.step()

        self._steps += 1
        if self._steps % self.target_freq == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        return loss.item()

    def decay_epsilon(self):
        self.epsilon = max(self.eps_end, self.epsilon * self.eps_decay)

print('DQNAgent definido.')


In [15]:
def train_dqn_cartpole(seed=0, n_episodes=N_EP_CARTPOLE,
                       lr=5e-4, gamma=0.99,
                       eps_start=1.0, eps_end=0.01, eps_decay=0.9995,
                       buf_size=20_000, batch=64, target_freq=200,
                       hidden=128, n_layers=2,
                       max_steps=MAX_STEPS_CARTPOLE, return_agent=False):
    set_global_seed(seed)
    env = gym.make('CartPole-v1')
    env.action_space.seed(seed)
    if hasattr(env.observation_space, 'seed'):
        env.observation_space.seed(seed)

    # Hiperparametros base DQN para CartPole: red compacta y target update frecuente.
    agent = DQNAgent(
        input_dim=env.observation_space.shape[0], n_actions=env.action_space.n,
        hidden=hidden, n_layers=n_layers, lr=lr, gamma=gamma,
        eps_start=eps_start, eps_end=eps_end, eps_decay=eps_decay,
        buf_size=buf_size, batch=batch, target_freq=target_freq
    )

    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=episode_seed(seed, ep))
        state  = obs.astype(np.float32)
        ep_ret, done, steps = 0.0, False, 0
        while not done and steps < max_steps:
            action = agent.select_action(state)
            next_obs, r, term, trunc, _ = env.step(action)
            done       = term or trunc
            next_state = next_obs.astype(np.float32)
            agent.store(state, action, r, next_state, done)
            agent.learn()
            agent.decay_epsilon()
            ep_ret += r
            state = next_state
            steps += 1
        returns.append(ep_ret)

    env.close()
    if return_agent:
        return returns, agent
    return returns


def train_dqn_tetris(seed=0, n_episodes=N_EP_TETRIS,
                     lr=1e-3, gamma=0.99,
                     eps_start=1.0, eps_end=0.05, eps_decay=0.999,
                     buf_size=20_000, batch=64, target_freq=300,
                     hidden=256, n_layers=2, max_steps=MAX_STEPS_TETRIS,
                     return_agent=False):
    set_global_seed(seed)
    from tetris_gymnasium.envs import Tetris as _TetrisEnvClass  # registra env
    env    = gym.make(TETRIS_ENV_ID)
    env.action_space.seed(seed)
    if hasattr(env.observation_space, 'seed'):
        env.observation_space.seed(seed)

    obs0, _ = env.reset(seed=episode_seed(seed, 0))
    in_dim  = int(_get_board(obs0).flatten().shape[0])
    # En Tetris se aumenta capacidad de red y se relaja target update por mayor complejidad.
    agent   = DQNAgent(
        input_dim=in_dim, n_actions=env.action_space.n,
        hidden=hidden, n_layers=n_layers, lr=lr, gamma=gamma,
        eps_start=eps_start, eps_end=eps_end, eps_decay=eps_decay,
        buf_size=buf_size, batch=batch, target_freq=target_freq
    )

    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=episode_seed(seed, ep))
        state  = _get_board(obs).flatten().astype(np.float32)
        ep_ret, done, steps = 0.0, False, 0
        while not done and steps < max_steps:
            action = agent.select_action(state)
            next_obs, r, term, trunc, _ = env.step(action)
            done       = term or trunc
            next_state = _get_board(next_obs).flatten().astype(np.float32)
            agent.store(state, action, r, next_state, done)
            agent.learn()
            agent.decay_epsilon()
            ep_ret += r
            state = next_state
            steps += 1
        returns.append(ep_ret)

    env.close()
    if return_agent:
        return returns, agent
    return returns


def evaluate_dqn_cartpole(agent, eval_seeds=EVAL_SEEDS_CARTPOLE,
                          max_steps=MAX_STEPS_CARTPOLE):
    env = gym.make('CartPole-v1')
    rets = []
    with torch.no_grad():
        for s in eval_seeds:
            obs, _ = env.reset(seed=s)
            state = obs.astype(np.float32)
            done, ep_ret, steps = False, 0.0, 0
            while not done and steps < max_steps:
                st = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                action = int(agent.q_net(st).argmax(1).item())
                next_obs, r, term, trunc, _ = env.step(action)
                done = term or trunc
                ep_ret += r
                state = next_obs.astype(np.float32)
                steps += 1
            rets.append(ep_ret)
    env.close()
    return np.array(rets, dtype=float)


def evaluate_dqn_tetris(agent, eval_seeds=EVAL_SEEDS_TETRIS,
                        max_steps=MAX_STEPS_TETRIS):
    from tetris_gymnasium.envs import Tetris as _TetrisEnvClass  # registra env
    env = gym.make(TETRIS_ENV_ID)
    rets = []
    with torch.no_grad():
        for s in eval_seeds:
            obs, _ = env.reset(seed=s)
            state = _get_board(obs).flatten().astype(np.float32)
            done, ep_ret, steps = False, 0.0, 0
            while not done and steps < max_steps:
                st = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                action = int(agent.q_net(st).argmax(1).item())
                next_obs, r, term, trunc, _ = env.step(action)
                done = term or trunc
                ep_ret += r
                state = _get_board(next_obs).flatten().astype(np.float32)
                steps += 1
            rets.append(ep_ret)
    env.close()
    return np.array(rets, dtype=float)


print('Funciones DQN definidas (train + eval fija).')


Funciones DQN definidas (train + eval fija).


### Cierre de Implementación DQN

Razón del diseño:

- `ReplayBuffer` + `TargetNetwork` para estabilidad del aprendizaje off-policy.
- Huber + clipping para robustez numérica.
- Misma política de semillas por episodio y misma evaluación fija que SARSA para comparabilidad.


### 2.1 DQN en CartPole-v1

Se replica el mismo protocolo de semillas que en SARSA (entrenamiento y evaluación).
Esto permite que cualquier diferencia observada se atribuya al algoritmo y no al muestreo.


In [ ]:
# Se entrena DQN en CartPole para todas las semillas de entrenamiento.
print(f'Entrenando DQN CartPole ({N_EP_CARTPOLE} ep x {len(SEEDS)} semillas)...')
dqn_cp_all, dqn_cp_eval = [], []
dqn_cp_demo_agent = None

for s in SEEDS:
    print(f'  seed={s}...', end=' ', flush=True)
    rets, agent = train_dqn_cartpole(seed=s, return_agent=True)
    eval_rets = evaluate_dqn_cartpole(agent)
    print(f'train-ult{W_FINAL_CARTPOLE}={np.mean(rets[-W_FINAL_CARTPOLE:]):.1f} | eval={np.mean(eval_rets):.1f}')
    dqn_cp_all.append(rets)
    dqn_cp_eval.append(eval_rets)
    if s == SEEDS[0]:
        dqn_cp_demo_agent = agent

dqn_cp_all  = np.array(dqn_cp_all)
dqn_cp_eval = np.array(dqn_cp_eval)
print('Listo.')


In [ ]:
# Se visualiza entrenamiento de DQN en CartPole y se reporta resumen train/eval.
fig, ax = plt.subplots(figsize=(9, 4))
plot_band(ax, dqn_cp_all, '#d62728', 'DQN')
ax.axhline(195, color='gray', linestyle='--', linewidth=0.9, label='Umbral (195)')
ax.axhline(500, color='green', linestyle=':', linewidth=0.9, label='Max (500)')
ax.set_xlabel('Episodio')
ax.set_ylabel(f'Retorno (MA-{SMOOTH_W})')
ax.set_title('DQN - CartPole-v1 (media +/- sigma, train)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Train media (ult.{W_FINAL_CARTPOLE} ep): {dqn_cp_all[:, -W_FINAL_CARTPOLE:].mean():.1f} +/- {dqn_cp_all[:, -W_FINAL_CARTPOLE:].std():.1f}')
print(f'Eval fija eps=0 (mismas seeds): {dqn_cp_eval.mean():.1f} +/- {dqn_cp_eval.std():.1f}')


### Lectura DQN CartPole (FULL)

Resultados observados en esta ejecución FULL:

- **Train final (ult. 50 episodios):** `265.1 ± 194.9`.
- **Eval fija eps=0 (agregada):** `222.4 ± 189.4`.
- Dispersión por semilla alta (ej.: seed=2 muy alta, seed=1/3 bajas).

Interpretación:

- DQN es inestable en este presupuesto/hyperparámetros para CartPole.
- Hay brecha respecto a SARSA tanto en media como en varianza.
- El resultado sugiere sensibilidad al seed y necesidad de afinado adicional (LR, target_freq,
  warmup del replay, scheduler de exploración, Double DQN).


### 2.2 DQN en Tetris

El tablero aplanado ($H \\times W$) alimenta directamente la red neuronal.
Mayor capacidad representacional que SARSA lineal, pero requiere más muestras.

In [ ]:
# Se entrena DQN en Tetris y luego se evalúa la política final de manera determinista.
dqn_tet_all, dqn_tet_eval = None, None
dqn_tet_demo_agent = None

if TETRIS_AVAILABLE:
    print(f'Entrenando DQN Tetris ({N_EP_TETRIS} ep x {len(SEEDS)} semillas)...')
    _tmp_train, _tmp_eval = [], []
    for s in SEEDS:
        print(f'  seed={s}...', end=' ', flush=True)
        rets, agent = train_dqn_tetris(seed=s, return_agent=True)
        eval_rets = evaluate_dqn_tetris(agent)
        print(f'train-ult{W_FINAL_TETRIS}={np.mean(rets[-W_FINAL_TETRIS:]):.3f} | eval={np.mean(eval_rets):.3f}')
        _tmp_train.append(rets)
        _tmp_eval.append(eval_rets)
        if s == SEEDS[0]:
            dqn_tet_demo_agent = agent

    dqn_tet_all  = np.array(_tmp_train)
    dqn_tet_eval = np.array(_tmp_eval)
    print('Listo.')
else:
    print('Tetris no disponible - celda omitida.')


In [ ]:
# Se visualiza entrenamiento de DQN en Tetris y se reporta resumen train/eval.
if dqn_tet_all is not None:
    fig, ax = plt.subplots(figsize=(9, 4))
    plot_band(ax, dqn_tet_all, '#d62728', 'DQN')
    ax.set_xlabel('Episodio')
    ax.set_ylabel(f'Retorno (MA-{SMOOTH_W})')
    ax.set_title('DQN - Tetris (media +/- sigma, train)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f'Train media (ult.{W_FINAL_TETRIS} ep): {dqn_tet_all[:, -W_FINAL_TETRIS:].mean():.3f} +/- {dqn_tet_all[:, -W_FINAL_TETRIS:].std():.3f}')
    print(f'Eval fija eps=0 (mismas seeds): {dqn_tet_eval.mean():.3f} +/- {dqn_tet_eval.std():.3f}')
else:
    print('Sin datos de Tetris.')


### Resultado DQN Tetris (FULL)

- **Train final (ult. 40 episodios):** `12.900 ± 2.577`.
- **Eval fija eps=0 (agregada):** `12.500 ± 2.787`.
- DQN supera a SARSA en media en Tetris, pero con varianza significativamente mayor.
- En Tetris, la mayor capacidad de DQN se traduce en mejor rendimiento medio.
- La dispersión confirma que aún hay inestabilidad y margen de mejora con más episodios
  y técnicas de estabilización (Double DQN, PER, CNN).


---
## 3. Estudio Comparativo SARSA vs. DQN (FULL ejecutado)

Comparación bajo condiciones controladas:

1. **Mismas semillas de entrenamiento** (`SEEDS`) para ambos algoritmos.
2. **Mismas semillas por episodio** (`episode_seed(seed, ep)`) en ambos algoritmos.
3. **Misma batería de evaluación fija** (`epsilon=0`) para ambos algoritmos y entornos.

- **CartPole:** SARSA domina claramente (`500.00`) frente a DQN (`265.08` train final, `222.40` eval).
- **Tetris:** DQN supera en media a SARSA (`12.90` vs `9.79` train final), con mayor varianza.

Se muestran curvas de entrenamiento (media móvil + dispersión) y evaluación final sin exploración.


In [ ]:
# Se agregan métricas de evaluación por semilla para comparar estabilidad entre algoritmos.
def _eval_seed_means(eval_arr):
    if eval_arr is None:
        return None
    return eval_arr.mean(axis=1)


def _plot_eval_bars(ax, means_a, means_b, labels=('SARSA', 'DQN')):
    vals = []
    errs = []
    for m in (means_a, means_b):
        if m is None:
            vals.append(np.nan)
            errs.append(0.0)
        else:
            vals.append(float(np.mean(m)))
            errs.append(float(np.std(m)))
    x = np.arange(len(labels))
    colors = ['#1f77b4', '#d62728']
    ax.bar(x, vals, yerr=errs, capsize=6, color=colors, alpha=0.85)
    ax.set_xticks(x, labels)
    ax.grid(True, axis='y', alpha=0.3)


n_envs = 1 + int(TETRIS_AVAILABLE and sarsa_tet_all is not None and dqn_tet_all is not None)
fig, axes = plt.subplots(2, n_envs, figsize=(8.5 * n_envs, 8.0))
if n_envs == 1:
    axes = np.array(axes).reshape(2, 1)

# Se comparan las curvas de entrenamiento en CartPole para ambos algoritmos.
ax = axes[0, 0]
plot_band(ax, sarsa_cp_all, '#1f77b4', 'SARSA semi-grad.')
plot_band(ax, dqn_cp_all,   '#d62728', 'DQN')
ax.axhline(195, color='gray', linestyle='--', linewidth=0.8, label='Umbral (195)')
ax.axhline(500, color='green', linestyle=':', linewidth=0.8, label='Max (500)')
ax.set_xlabel('Episodio'); ax.set_ylabel(f'Retorno train (MA-{SMOOTH_W})')
ax.set_title('CartPole-v1 - entrenamiento')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Se compara el desempeño de política final en CartPole con evaluación fija sin exploración.
ax = axes[1, 0]
_plot_eval_bars(ax, _eval_seed_means(sarsa_cp_eval), _eval_seed_means(dqn_cp_eval))
ax.set_title('CartPole-v1 - eval fija eps=0 (media +/- std entre semillas)')
ax.set_ylabel('Retorno eval')

if n_envs == 2:
    # Se comparan las curvas de entrenamiento en Tetris para ambos algoritmos.
    ax = axes[0, 1]
    plot_band(ax, sarsa_tet_all, '#1f77b4', 'SARSA semi-grad.')
    plot_band(ax, dqn_tet_all,   '#d62728', 'DQN')
    ax.set_xlabel('Episodio'); ax.set_ylabel(f'Retorno train (MA-{SMOOTH_W})')
    ax.set_title('Tetris - entrenamiento')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # Se compara el desempeño de política final en Tetris con evaluación fija sin exploración.
    ax = axes[1, 1]
    _plot_eval_bars(ax, _eval_seed_means(sarsa_tet_eval), _eval_seed_means(dqn_tet_eval))
    ax.set_title('Tetris - eval fija eps=0 (media +/- std entre semillas)')
    ax.set_ylabel('Retorno eval')

plt.tight_layout(); plt.show()


In [ ]:
# Se consolida una tabla final con métricas de entrenamiento y evaluación por entorno.
def summary_row(algo, env_name, train_arr, eval_arr, w):
    if train_arr is None:
        return {
            'Algoritmo': algo,
            'Entorno': env_name,
            'Ventana final': str(w),
            'Train media final': 'N/A',
            'Train std final': 'N/A',
            'Mejor ep (MA)': 'N/A',
            'Eval eps=0 media': 'N/A',
            'Eval eps=0 std': 'N/A',
            'Eval n_seeds': 'N/A',
            'Eval n_episodes': 'N/A',
        }

    final = train_arr[:, -w:]
    sm  = np.array([smooth(r, SMOOTH_W) for r in train_arr])
    n   = min(len(s) for s in sm)
    sm  = np.vstack([s[:n] for s in sm])
    best = int(sm.mean(0).argmax())

    if eval_arr is None:
        eval_mean, eval_std, n_seeds, n_eval = np.nan, np.nan, 0, 0
    else:
        eval_seed_means = eval_arr.mean(axis=1)
        eval_mean = float(eval_seed_means.mean())
        eval_std  = float(eval_seed_means.std())
        n_seeds, n_eval = int(eval_arr.shape[0]), int(eval_arr.shape[1])

    return {
        'Algoritmo': algo,
        'Entorno': env_name,
        'Ventana final': str(w),
        'Train media final': f'{final.mean():.2f}',
        'Train std final': f'{final.std():.2f}',
        'Mejor ep (MA)': str(best),
        'Eval eps=0 media': f'{eval_mean:.2f}' if np.isfinite(eval_mean) else 'N/A',
        'Eval eps=0 std': f'{eval_std:.2f}' if np.isfinite(eval_std) else 'N/A',
        'Eval n_seeds': str(n_seeds),
        'Eval n_episodes': str(n_eval),
    }
# Pasar datos a DataFrame para visualización tabular    
tabla = pd.DataFrame([
    summary_row('SARSA semi-grad.', 'CartPole-v1', sarsa_cp_all, sarsa_cp_eval, W_FINAL_CARTPOLE),
    summary_row('DQN',              'CartPole-v1', dqn_cp_all,   dqn_cp_eval,   W_FINAL_CARTPOLE),
    summary_row('SARSA semi-grad.', 'Tetris',      sarsa_tet_all, sarsa_tet_eval, W_FINAL_TETRIS),
    summary_row('DQN',              'Tetris',      dqn_tet_all,   dqn_tet_eval,   W_FINAL_TETRIS),
])

print('Tabla comparativa final (entrenamiento + eval fija eps=0):')
print('Nota: eval usa exactamente el mismo conjunto de semillas para SARSA y DQN en cada entorno.')
print(tabla.to_string(index=False))


---
## 4. Discusión y Conclusiones

### 4.1 Protocolo de comparabilidad y reproducibilidad

La comparación SARSA vs. DQN se hizo bajo protocolo controlado:

1. Mismo conjunto de semillas de entrenamiento (`SEEDS=[0,1,2,3,4]`).
2. Misma semilla por episodio (`episode_seed(seed, ep)`) en ambos algoritmos.
3. Evaluación final con `epsilon=0` usando exactamente las mismas semillas de evaluación.

Esto reduce sesgos por aleatoriedad y hace defendible la comparación técnica.

### 4.2 Resultados cuantitativos clave (FULL)

| Algoritmo | Entorno | Train final | Eval eps=0 | Variabilidad |
|-----------|---------|-------------|------------|--------------|
| SARSA | CartPole | **500.00 ± 0.00** | **500.00 ± 0.00** | Muy baja |
| DQN | CartPole | 265.08 ± 194.87 | 222.40 ± 168.52 | Muy alta |
| SARSA | Tetris | 9.79 ± 0.54 | 9.95 ± 0.00* | Baja |
| DQN | Tetris | **12.90 ± 2.58** | **12.50 ± 1.89** | Media/alta |

\* En tabla el `std` es sobre medias por semilla; en la salida agregada por episodios de evaluación aparece dispersión interna.

### 4.3 Interpretación por entorno

**CartPole-v1**

- SARSA con Fourier alcanza solución óptima y estable.
- DQN no consolida convergencia robusta con esta configuración (alta sensibilidad a semilla).

**Tetris**

- SARSA presenta comportamiento estable pero limitado por capacidad representacional.
- DQN mejora media de rendimiento, consistente con su mayor expresividad, aunque con varianza mayor.

1. La “mejor técnica” depende del entorno y de la estabilidad del entrenamiento, no solo del promedio final.
3. En problemas complejos como Tetris, DQN necesita más presupuesto y técnicas de estabilización para explotar su potencial.

### 4.5 Future Work

- CartPole DQN: ajustar `lr`, `target_freq`, `eps_decay`, `learning_starts` y probar Double DQN.
- Tetris DQN: aumentar episodios y pasar a arquitectura CNN + PER.
- Reporte final: añadir IC95% por bootstrap o más semillas (10+).


---
## 5. Simulación de políticas entrenadas

Se generan animaciones con política greedy (`epsilon=0`) usando la primera semilla entrenada.

- `SARSA CartPole demo return`: **500.0**
- `DQN CartPole demo return`: **109.0**
- `SARSA Tetris demo return`: **10.0**
- `DQN Tetris demo return`: **7.0**

In [ ]:
# Se inicializan referencias por defecto para permitir ejecución aislada de esta celda.
sarsa_cp_demo_agent = globals().get('sarsa_cp_demo_agent', None)
sarsa_cp_demo_feat  = globals().get('sarsa_cp_demo_feat', None)
dqn_cp_demo_agent   = globals().get('dqn_cp_demo_agent', None)
sarsa_tet_demo_agent = globals().get('sarsa_tet_demo_agent', None)
dqn_tet_demo_agent   = globals().get('dqn_tet_demo_agent', None)

# Se transforma una secuencia de frames en una animación incrustada en notebook.
def _animate_frames(frames, title='Simulacion', fps=20, max_frames=250):
    if frames is None or len(frames) == 0:
        print(f'{title}: no se pudieron obtener frames.')
        return
    frames = frames[:max_frames]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis('off')
    ax.set_title(title)
    im = ax.imshow(frames[0])

    def _update(i):
        im.set_data(frames[i])
        return [im]

    ani = animation.FuncAnimation(
        fig, _update, frames=len(frames), interval=1000 / fps, blit=True
    )
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

# Se ejecuta un episodio de CartPole capturando frames para visualizar la política aprendida.
def rollout_cartpole(policy_fn, seed=123, max_steps=MAX_STEPS_CARTPOLE):
    env = gym.make('CartPole-v1', render_mode='rgb_array')
    obs, _ = env.reset(seed=seed)
    frames, ret = [], 0.0

    frame = env.render()
    if frame is not None:
        frames.append(frame)

    done, steps = False, 0
    while not done and steps < max_steps:
        a = int(policy_fn(obs))
        obs, r, term, trunc, _ = env.step(a)
        done = term or trunc
        ret += r
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        steps += 1

    env.close()
    return frames, ret

# Se ejecuta un episodio de Tetris capturando frames cuando el entorno los ofrece.
def rollout_tetris(policy_fn, seed=456, max_steps=MAX_STEPS_TETRIS):
    try:
        env = gym.make(TETRIS_ENV_ID, render_mode='rgb_array')
    except TypeError:
        env = gym.make(TETRIS_ENV_ID)

    obs, _ = env.reset(seed=seed)
    frames, ret = [], 0.0

    try:
        frame = env.render()
        if frame is not None and hasattr(frame, 'shape'):
            frames.append(frame)
    except Exception:
        pass

    done, steps = False, 0
    while not done and steps < max_steps:
        a = int(policy_fn(obs))
        obs, r, term, trunc, _ = env.step(a)
        done = term or trunc
        ret += r
        try:
            frame = env.render()
            if frame is not None and hasattr(frame, 'shape'):
                frames.append(frame)
        except Exception:
            pass
        steps += 1

    env.close()
    return frames, ret

SHOW_SIMULATIONS = True

if SHOW_SIMULATIONS:
    # Se visualiza la política SARSA entrenada en CartPole.
    if sarsa_cp_demo_agent is not None:
        def _pol_sarsa_cp(obs):
            phi = sarsa_cp_demo_feat(obs)
            return np.argmax(sarsa_cp_demo_agent.q_values(phi))

        fr, ret = rollout_cartpole(_pol_sarsa_cp, seed=999)
        print(f'SARSA CartPole demo return: {ret:.1f}')
        _animate_frames(fr, title='SARSA - CartPole', fps=20)

    # Se visualiza la política DQN entrenada en CartPole.
    if dqn_cp_demo_agent is not None:
        @torch.no_grad()
        def _pol_dqn_cp(obs):
            st = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return dqn_cp_demo_agent.q_net(st).argmax(1).item()

        fr, ret = rollout_cartpole(_pol_dqn_cp, seed=999)
        print(f'DQN CartPole demo return: {ret:.1f}')
        _animate_frames(fr, title='DQN - CartPole', fps=20)

    # Se visualiza la política SARSA entrenada en Tetris.
    if TETRIS_AVAILABLE and sarsa_tet_demo_agent is not None:
        def _pol_sarsa_tet(obs):
            phi = tetris_featurize(obs)
            return np.argmax(sarsa_tet_demo_agent.q_values(phi))

        fr, ret = rollout_tetris(_pol_sarsa_tet, seed=1999)
        print(f'SARSA Tetris demo return: {ret:.1f}')
        _animate_frames(fr, title='SARSA - Tetris', fps=12)

    # Se visualiza la política DQN entrenada en Tetris.
    if TETRIS_AVAILABLE and dqn_tet_demo_agent is not None:
        @torch.no_grad()
        def _pol_dqn_tet(obs):
            st_np = _get_board(obs).flatten().astype(np.float32)
            st = torch.tensor(st_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return dqn_tet_demo_agent.q_net(st).argmax(1).item()

        fr, ret = rollout_tetris(_pol_dqn_tet, seed=1999)
        print(f'DQN Tetris demo return: {ret:.1f}')
        _animate_frames(fr, title='DQN - Tetris', fps=12)
